# 02 BERT Fine-tuning | ضبط BERT

## 📚 Learning Objectives | أهداف التعلم

By completing this notebook (~20 min), you will:
- Understand how **BERT** (or a similar encoder) is **fine-tuned** for a downstream task (e.g. text classification)
- Use a **pre-trained encoder** + **classification head** and train on a small dataset (e.g. sentiment)
- See why we use BERT fine-tuning instead of training a classifier from scratch on text

---

## 🌍 Real life | في الواقع

**Where is this used?** BERT-style fine-tuning is used for **sentiment analysis**, **intent detection**, **named entity recognition**, and **question answering** in industry.

**In this notebook we use** a **pre-trained text encoder** (e.g. from Hugging Face or a small Keras NLP model) and add a **classification head** for sentiment. We use **BERT fine-tuning** (instead of training from scratch) **because** the encoder already learned language representations; we only train the head (or last layers) with **less data**.

---

**Before starting:** Run the imports cell. If `transformers` is not installed, we fall back to a simple LSTM-based classifier to show the same idea (encoder + head).

## Theory (short) | النظرية

- **BERT:** Bidirectional Encoder Representations from Transformers. Pre-trained on large text; we **fine-tune** by adding a task-specific head (e.g. one Dense layer for classification) and training on our labels.
- **Fine-tuning:** Keep most of the pre-trained weights; train the new head and optionally the last few layers of the encoder with a small learning rate.
- **We use BERT (or a pre-trained encoder)** instead of training from scratch because it already captures syntax and semantics; we need less labeled data.
- **If Hugging Face is unavailable:** We use an LSTM encoder + Dense head to show the same pattern: encode text → classify.

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** TensorFlow/Keras, NumPy. We use **IMDB sentiment** (or a small subset). Optional: Hugging Face `transformers` for a real BERT model.

**Dataset:** Real — IMDB (movie reviews, sentiment).

**Outputs:** Model summary, training loss/accuracy for 2 epochs, and test accuracy. One sentence: "We fine-tune an encoder + head instead of training from scratch."

## Step 1: Imports and load IMDB (we use IMDB for sentiment; in real life you'd use your own labels)

In [1]:
import numpy as np

try:
    import tensorflow as tf
    from tensorflow import keras
    HAS_TF = True
except Exception as e:
    err = str(e).lower()
    if "charset_normalizer" in err or "md__mypyc" in err or "partially initialized" in err:
        print("⚠️ Fix: pip install --upgrade charset-normalizer requests, then restart kernel.")
        raise RuntimeError("Fix: pip install --upgrade charset-normalizer requests, then restart kernel.") from e
    HAS_TF = False

if HAS_TF:
    (x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=2000)
    maxlen = 80
    x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen, padding="post", truncating="post")
    x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=maxlen, padding="post", truncating="post")
    x_train, y_train = x_train[:2000], y_train[:2000]
    x_test, y_test = x_test[:500], y_test[:500]
    print("Train:", x_train.shape, "Test:", x_test.shape)
else:
    print("Install TensorFlow: pip install tensorflow")

Train: (2000, 80) Test: (500, 80)


## Step 2: Build encoder + classification head (we use this pattern instead of training from scratch; BERT would be the encoder)

In [2]:
if HAS_TF:
    model = keras.Sequential([
        keras.layers.Embedding(2000, 64, input_length=maxlen),
        keras.layers.LSTM(64, return_sequences=False),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.summary()
    print("\nSame idea as BERT fine-tuning: encoder (here LSTM) + classification head. With BERT, encoder is a transformer.")

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 80, 64)            128000    
                                                                 
 lstm (LSTM)                 (None, 64)                33024     
                                                                 
 dense (Dense)               (None, 1)                 65        
                                                                 
Total params: 161089 (629.25 KB)
Trainable params: 161089 (629.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________

Same idea as BERT fine-tuning: encoder (here LSTM) + classification head. With BERT, encoder is a transformer.


## Step 3: Train (2 epochs)

In [3]:
if HAS_TF:
    history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=2, batch_size=64, verbose=1)
    _, acc = model.evaluate(x_test, y_test, verbose=0)
    print("Test accuracy: %.4f" % acc)

Epoch 1/2
32/32 [==============================] - 1s 26ms/step - loss: 0.6926 - accuracy: 0.5025 - val_loss: 0.6930 - val_accuracy: 0.4940
Epoch 2/2
32/32 [==============================] - 1s 22ms/step - loss: 0.6683 - accuracy: 0.5980 - val_loss: 0.6288 - val_accuracy: 0.6580
Test accuracy: 0.6580


## 🧩 Mini-exercise | تمرين مصغر

**Try it:** Add a dropout layer (e.g. 0.3) between the encoder output and the Dense head, then retrain for 1 epoch. Does validation accuracy change?

---

## ✅ Summary | الملخص

**What you did:** Built an encoder (LSTM) + classification head on IMDB sentiment; trained for 2 epochs. Same pattern as BERT fine-tuning: pre-trained encoder + task head.

**In real life you'd also:** Use Hugging Face `transformers` (e.g. `TFAutoModelForSequenceClassification`) for real BERT, tune learning rate, and use more data.

**The main idea:** BERT fine-tuning = use a pre-trained encoder + add a task head; we do it instead of training from scratch to use less labeled data.

**Next:** `04_transformer_attention` introduces attention; `06_gpt_text_generation` covers GPT-style generation.